In [ ]:
!pip -q install textblob vaderSentiment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

In [ ]:
texts = [
    "I absolutely love this phone, the camera is amazing!",
    "This is the worst service I have ever experienced.",
    "The movie was okay, nothing special.",
    "The food was GREAT!!! Will definitely come again :)",
    "I do not like the new update at all.",
    "The delivery was not bad.",
    "Battery life is terrible but the screen is gorgeous.",
    "The meeting is scheduled for 3 PM tomorrow.",
]

In [ ]:
def tb_label(p): return "Positive" if p > 0.05 else "Negative" if p < -0.05 else "Neutral"
tb = [TextBlob(t).sentiment for t in texts]
for t, s in zip(texts, tb):
    print(f"{s.polarity:+.2f}  subj={s.subjectivity:.2f}  {tb_label(s.polarity):8s} | {t}")

In [ ]:
def vd_label(c): return "Positive" if c >= 0.05 else "Negative" if c <= -0.05 else "Neutral"
vd = [vader.polarity_scores(t) for t in texts]
for t, s in zip(texts, vd):
    print(f"{s['compound']:+.3f}  neg={s['neg']:.2f} neu={s['neu']:.2f} pos={s['pos']:.2f}  {vd_label(s['compound']):8s} | {t}")

In [ ]:
df = pd.DataFrame({
    "text": texts,
    "TextBlob_polarity": [s.polarity for s in tb],
    "TextBlob_label": [tb_label(s.polarity) for s in tb],
    "VADER_compound": [s["compound"] for s in vd],
    "VADER_label": [vd_label(s["compound"]) for s in vd],
})
df["agree"] = df.TextBlob_label == df.VADER_label
pd.set_option("display.max_colwidth", 60)
display(df)
print("Agreement:", df.agree.mean().round(2))

df.set_index(df.index + 1)[["TextBlob_polarity", "VADER_compound"]].plot(kind="bar", figsize=(8, 4))
plt.axhline(0, color="k", lw=0.8); plt.xlabel("Sentence no."); plt.title("TextBlob vs VADER"); plt.show()